# Laboratorio 7: Pipeline ETL, Profiling y Gobernanza en el Data Lake
**Materia:** Fundamentos de Big Data (IDSD413) - EPN

**Objetivo:** Simular un escenario real de ingesta de datos ruidosos (anomalías, nulos, duplicados), ejecutar un perfilamiento estadístico para dimensionar el caos, y procesarlos a través de un flujo ETL construyendo las capas lógicas de un Data Lake (Capa *Raw* y Capa *Curated*).



In [ ]:
import pandas as pd
import numpy as np
import os

print("1/3 Generando datos base perfectos en memoria...")
np.random.seed(42)
N_ROWS = 5_000_000
df_base = pd.DataFrame({
    'id_transaccion': np.arange(N_ROWS),
    'id_cliente': np.random.randint(1, 100000, N_ROWS),
    'monto': np.random.uniform(5.0, 2500.0, N_ROWS),
    'categoria': np.random.choice(['Electronica', 'Ropa', 'Hogar', 'Alimentos', 'Juguetes'], N_ROWS),
    'estado': np.random.choice(['Completado', 'Pendiente', 'Cancelado', 'Reembolsado'], N_ROWS)
})

# Guardamos una base fresca para evitar el error de bytes corruptos en Colab
nombre_fresco = 'base_fresca.parquet'
df_base.to_parquet(nombre_fresco, engine='pyarrow')

print("2/3 Leyendo archivo desde disco e inyectando caos (Simulación de ingesta real)...")
df_original = pd.read_parquet(nombre_fresco, engine='pyarrow')




# a) Inyectamos un 5% de nulos
n_nulos = int(len(df_original) * 0.05)
indices_nulos = np.random.choice(df_original.index, n_nulos, replace=False)
df_original.loc[indices_nulos, 'monto'] = np.nan

# b) Duplicamos 50,000 registros
df_duplicados = df_original.sample(n=50000)
df_ingesta = pd.concat([df_original, df_duplicados], ignore_index=True)

# c) Inyectamos outliers (valores absurdos que rompen la estadística)
df_ingesta.loc[0:100, 'monto'] = 9999999.99

print(f"3/3 ¡Éxito! Dataset de ingesta listo. Total de filas: {len(df_ingesta):,}")

## Fase 1: Profiling y Capa RAW (El Diagnóstico del Caos)
Antes de transformar, debemos perfilar el archivo crudo para entender la magnitud del problema y luego guardarlo en la capa inmutable del Data Lake.

In [ ]:
print("--- DIAGNÓSTICO ESTADÍSTICO (PROFILING) ---\n")

# 1. Auditoría de Nulos y Duplicados
print("1. Valores Nulos por Columna:")
print(df_ingesta.isnull().sum())
print(f"\n2. Registros Duplicados Exactos: {df_ingesta.duplicated().sum():,}")

# 2. Auditoría de Outliers (Fíjense en la diferencia entre el promedio y el max)
print("\n3. Resumen Estadístico de 'monto':")
print(df_ingesta['monto'].describe().apply(lambda x: f"{x:,.2f}"))

# 3. Guardado en Capa RAW (Inmutable)
os.makedirs('./datalake/raw', exist_ok=True)
os.makedirs('./datalake/curated', exist_ok=True)

print("\n--- EXTRACT & LOAD (EL) ---")
print("Guardando evidencia en la Capa RAW...")
df_ingesta.to_parquet('./datalake/raw/ingesta_transacciones_sucio.parquet')
print("¡Archivo RAW almacenado con éxito! Listos para limpiar.")

## Fase 2: Transform (T) - Pipeline ETL de Limpieza y Normalización
Levantamos el dato desde RAW y ejecutamos limpieza masiva: eliminar duplicados, tratar nulos mediante imputación robusta (mediana) y aislar valores atípicos (outliers) utilizando el Rango Intercuartílico (IQR). Finalmente, normalizamos para Machine Learning.

In [ ]:
import os

print("Iniciando Transformación...\n")
df_ingesta = pd.read_parquet('./datalake/raw/ingesta_transacciones_sucio.parquet')

# 1. Eliminación de Duplicados
antes = len(df_ingesta)
df_limpio = df_ingesta.drop_duplicates().copy()
print(f"Duplicados eliminados: {antes - len(df_limpio):,}")

# 2. Reto 2: Lógica de Negocio - Descarte Inteligente e Imputación de Nulos
# Eliminar transacciones con estado 'Cancelado' y monto nulo
filtro_cancelado_nulo = (df_limpio['estado'] == 'Cancelado') & (df_limpio['monto'].isnull())
eliminados_cancelados = filtro_cancelado_nulo.sum()
df_limpio = df_limpio[~filtro_cancelado_nulo].copy()
print(f"Transacciones 'Cancelado' con monto nulo eliminadas: {eliminados_cancelados:,}")

# Imputación robusta de Nulos restantes (Mediana por categoría)
estados_permitidos = ['Completado', 'Pendiente', 'Reembolsado']
medianas = df_limpio.groupby('categoria')['monto'].transform('median')
mascara_imputacion = df_limpio['estado'].isin(estados_permitidos) & df_limpio['monto'].isnull()
df_limpio.loc[mascara_imputacion, 'monto'] = medianas.loc[mascara_imputacion]
print(f"Nulos restantes en monto: {df_limpio['monto'].isnull().sum()}")

# 3. Reto 1: Tratamiento de Outliers y Capa de Cuarentena (Regla IQR)
os.makedirs('./datalake/quarantine', exist_ok=True)

Q1 = df_limpio['monto'].quantile(0.25)
Q3 = df_limpio['monto'].quantile(0.75)
IQR = Q3 - Q1
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

# Aislar y guardar los outliers en formato Parquet
mascara_outliers = (df_limpio['monto'] < limite_inferior) | (df_limpio['monto'] > limite_superior)
df_outliers = df_limpio.loc[mascara_outliers].copy()
df_outliers.to_parquet('./datalake/quarantine/outliers_monto.parquet', index=False)

# Filtrar df_limpio conservando únicamente los registros válidos
df_limpio = df_limpio.loc[~mascara_outliers].copy()
print(f"Outliers enviados a cuarentena: {len(df_outliers):,}")

# 4. Normalización Min-Max (Escala de 0 a 1)
min_monto = df_limpio['monto'].min()
max_monto = df_limpio['monto'].max()

df_limpio['monto_escalado'] = (df_limpio['monto'] - min_monto) / (max_monto - min_monto)
print("Normalización Min-Max completada.")

## Fase 3: Load (Curated Layer) y Verificación Final
Guardamos el dato limpio en formato columnar en la capa *Curated*. Finalmente, comparamos la estadística del dato curado frente al desastre original.

In [ ]:
# 1. Guardamos en la capa analítica
df_limpio.to_parquet('./datalake/curated/transacciones_model_ready.parquet')
print("Datos escritos en la Capa CURATED exitosamente.\n")

# 2. Demostración del dato limpio (El "Después")
print("--- ESTADÍSTICA DEL DATO CURADO (Monto limpio vs Escalado) ---")
# Comparamos la estadística final para demostrar que la media volvió a la normalidad
print(df_limpio[['monto', 'monto_escalado']].describe().map(lambda x: f"{x:,.2f}"))

print("\n--- MUESTRA DE DATOS LISTOS PARA MACHINE LEARNING ---")
display(df_limpio[['id_transaccion', 'categoria', 'monto', 'monto_escalado']].head(5))

print("\n--- RESUMEN DE ALMACENAMIENTO DEL DATA LAKE ---")
print(f"RAW (Sucio): {os.path.getsize('./datalake/raw/ingesta_transacciones_sucio.parquet') / (1024*1024):.2f} MB")
print(f"CURATED (Limpio): {os.path.getsize('./datalake/curated/transacciones_model_ready.parquet') / (1024*1024):.2f} MB")